# Notebook 07 — AoU preparation for BioMe external validation

This notebook creates the **BioMe-compatible AoU models** that will be frozen and sent with the final BioMe extraction/scoring package.

The design is intentionally one-directional:

**AoU TRAIN → model tuning and fitting → AoU CAL → isotonic calibration and fixed-policy thresholds → AoU TEST → internal performance audit → freeze model package → BioMe external validation.**

BioMe data are **not** used for model training, hyperparameter tuning, calibration, feature selection, or threshold selection.

No figures are generated in this notebook. All AoU performance is reported in tables at the end.

## Analysis contract

- Uses the corrected AoU landmark cohorts and frozen 60/20/20 split from Notebooks 03A/03B.
- Starts from the candidate AoU–BioMe mutual feature list embedded in the previously shared BioMe validation script.
- Defines each BioMe-compatible model as the **corrected AoU model-specific feature set intersected with the candidate BioMe mutual feature set**, preserving the corrected AoU feature order.
- Trains primary XGBoost models at 30d, 6m, 1y, and 2y: M1, M2, M3, and M4a/M4 as appropriate.
- Re-runs 20-trial Optuna tuning on AoU TRAIN because restricting the feature set changes the optimization problem.
- Fits isotonic calibration on AoU CAL.
- Learns thresholds on AoU CAL at target specificity 0.80, 0.85, 0.90, and 0.95; these thresholds are later applied unchanged in BioMe.
- Also prepares three one-year secondary models so BioMe does not need another scoring rerun later: a four-variable clinical logistic model, logistic regression on mutual M4, and random forest on mutual M4.
- Reports AoU TEST discrimination, calibration, fixed-policy classification performance, raw-versus-calibrated discrimination, and the performance difference between full corrected AoU models and the BioMe-compatible mutual-feature models.

In [ ]:
import json
import os
import pickle
import warnings

import gcsfs
import numpy as np
import optuna
import pandas as pd
import xgboost as xgb

from scipy.special import expit, logit
from scipy.optimize import brentq

from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ------------------------------------------------------------------
# Project paths
# ------------------------------------------------------------------
migration_folder = "rw-migration-aou-rw-639fade3"
bucket = migration_folder

COHORT_DIR = f"{bucket}/aou_dkd/dkd_cohort_20260821"
PRIMARY_MODEL_DIR = f"{bucket}/aou_dkd/dkd_models_20260822"
PRIMARY_RESULT_DIR = f"{bucket}/aou_dkd/dkd_model_results_20260822"

EXTERNAL_MODEL_DIR = f"{bucket}/aou_dkd/dkd_models_biome_20260823"
EXTERNAL_RESULT_DIR = f"{bucket}/aou_dkd/dkd_biome_prep_20260823"

fs = gcsfs.GCSFileSystem()

ID_COL = "person_id"
Y_COL = "dkd_binary"
SOURCE_DATE = "20260822"
RUN_DATE = "20260823"
SEED = 42
CV_FOLDS = 5
N_XGB_TRIALS = 20
N_LR_TRIALS = 20
N_RF_TRIALS = 20
N_BOOT = 2000
BOOT_SEED = 42
SPEC_LEVELS = [0.80, 0.85, 0.90, 0.95]
PRIMARY_SPEC = 0.90

# Secondary one-year models are included now to avoid asking BioMe
# to rerun scoring later for likely reviewer analyses.
RUN_1Y_SECONDARY_MODELS = True

# Parallelism tuned for the large AoU workspace used in prior notebooks.
XGB_N_JOBS = 8
CV_N_JOBS = 5
RF_CV_N_JOBS = 16
RF_FINAL_N_JOBS = 64

XGB_BOUNDS = {
    "gamma": (1e-8, 1.0),
    "max_depth": (3, 6),
    "min_child_weight": (1, 6),
    "subsample": (0.6, 1.0),
    "colsample_bytree": (0.6, 1.0),
    "learning_rate": (0.01, 0.15),
    "n_estimators": (150, 300),
    "reg_alpha": (1e-8, 0.5),
    "reg_lambda": (1e-8, 0.5),
}

PRIMARY_EXTERNAL_MODEL_ORDER = {
    "30d": ["M1", "M2", "M3", "M4a"],
    "6m":  ["M1", "M2", "M3", "M4a"],
    "1y":  ["M1", "M2", "M3", "M4"],
    "2y":  ["M1", "M2", "M3", "M4"],
}

print("Notebook 07 configuration loaded")
print("External model directory:", EXTERNAL_MODEL_DIR)
print("External result directory:", EXTERNAL_RESULT_DIR)

In [ ]:
def read_gcs_pickle(path):
    with fs.open(path, "rb") as f:
        return pd.read_pickle(f)


def read_gcs_csv(path):
    with fs.open(path, "r") as f:
        return pd.read_csv(f)


def save_gcs_pickle(obj, path):
    with fs.open(path, "wb") as f:
        pickle.dump(obj, f)
    print("Saved pickle:", path)


def save_gcs_csv(df, path):
    with fs.open(path, "w") as f:
        df.to_csv(f, index=False)
    print("Saved CSV:", path)


def save_gcs_json(obj, path):
    with fs.open(path, "w") as f:
        json.dump(obj, f, indent=2)
    print("Saved JSON:", path)


def basename(path):
    return str(path).rstrip("/").split("/")[-1]

## Candidate mutual feature list from the previously shared BioMe scoring script

The old BioMe Python script contained **342 entries total: 340 candidate predictors plus `person_id` and `dkd_binary`**. We use that list only as the starting compatibility inventory. The final model-specific feature sets are generated below by intersecting it with the corrected AoU feature manifest.

This notebook therefore does **not** revive features removed by corrected AoU curation, and it does not create artificial missing categories.

In [ ]:
CANDIDATE_BIOME_MUTUAL_FEATURES = ['person_id', 'dkd_binary', 'age_at_t2d', 'sex_male', 'PGS000036_T2D_PRS', 'PGS000039_Stroke_PRS', 'PGS000304_HbA1c_PRS', 'PGS000305_FG_PRS', 'PGS000682_eGFR_PRS', 'PGS000822_UACR_PRS', 'PGS000884_eGFR_PRS', 'PGS000888_LDL_PRS', 'PGS002126_Smoking_PRS', 'PGS002724_Stroke_PRS', 'PGS003309_eGFR_CysC_PRS', 'PGS003469_HOMAB_PRS', 'PGS003470_HOMAIR_PRS', 'PGS003962_DBP_PRS', 'PGS003966_SBP_PRS', 'PGS004236_Hypertension_PRS', 'PGS004888_CAD_PRS', 'PGS004889_CKD_PRS', 'PGS004896_Microalbuminuria_PRS', 'PGS005198_BMI_PRS', 'PGS005313_Afib_PRS', 'ALP_last', 'ALP_max', 'ALP_mean', 'ALP_min', 'ALP_n', 'ALT_last', 'ALT_max', 'ALT_mean', 'ALT_min', 'ALT_n', 'AST_last', 'AST_max', 'AST_mean', 'AST_min', 'AST_n', 'Albumin_last', 'Albumin_max', 'Albumin_mean', 'Albumin_min', 'Albumin_n', 'BUN_last', 'BUN_max', 'BUN_mean', 'BUN_min', 'BUN_n', 'Cl_last', 'Cl_max', 'Cl_mean', 'Cl_min', 'Cl_n', 'DirectBilirubin_last', 'DirectBilirubin_max', 'DirectBilirubin_mean', 'DirectBilirubin_min', 'DirectBilirubin_n', 'eGFR_last', 'eGFR_max', 'eGFR_mean', 'eGFR_min', 'eGFR_n', 'HbA1c_last', 'HbA1c_max', 'HbA1c_mean', 'HbA1c_min', 'HbA1c_n', 'HCO3_last', 'HCO3_max', 'HCO3_mean', 'HCO3_min', 'HCO3_n', 'HDL_last', 'HDL_max', 'HDL_mean', 'HDL_min', 'HDL_n', 'Hgb_last', 'Hgb_mean', 'Hgb_min', 'Hgb_n', 'K_last', 'K_max', 'K_mean', 'K_min', 'K_n', 'LDL_last', 'LDL_max', 'LDL_mean', 'LDL_min', 'LDL_n', 'Na_last', 'Na_max', 'Na_mean', 'Na_min', 'Na_n', 'RBG_last', 'RBG_max', 'RBG_mean', 'RBG_min', 'RBG_n', 'Screatinine_last', 'Screatinine_max', 'Screatinine_mean', 'Screatinine_min', 'Screatinine_n', 'TG_last', 'TG_max', 'TG_mean', 'TG_min', 'TG_n', 'TotalBilirubin_last', 'TotalBilirubin_max', 'TotalBilirubin_mean', 'TotalBilirubin_min', 'TotalBilirubin_n', 'TotalChol_last', 'TotalChol_max', 'TotalChol_mean', 'TotalChol_min', 'TotalChol_n', 'UACR_last', 'UACR_max', 'UACR_mean', 'UACR_min', 'UACR_n', 'UrAlb_last', 'UrAlb_max', 'UrAlb_mean', 'UrAlb_min', 'UrAlb_n', 'UricAcid_last', 'UricAcid_max', 'UricAcid_mean', 'UricAcid_min', 'UricAcid_n', 'ALP_ever_abnormal', 'ALP_frac_abnormal', 'ALP_recency_days', 'ALT_ever_abnormal', 'ALT_frac_abnormal', 'ALT_recency_days', 'AST_ever_abnormal', 'AST_frac_abnormal', 'AST_recency_days', 'Albumin_ever_abnormal', 'Albumin_frac_abnormal', 'Albumin_recency_days', 'BUN_ever_abnormal', 'BUN_frac_abnormal', 'BUN_recency_days', 'Cl_ever_abnormal', 'Cl_frac_abnormal', 'Cl_recency_days', 'DirectBilirubin_ever_abnormal', 'DirectBilirubin_frac_abnormal', 'DirectBilirubin_recency_days', 'eGFR_cov', 'eGFR_ever_abnormal', 'eGFR_frac_abnormal', 'eGFR_recency_days', 'eGFR_sd', 'eGFR_slope', 'eGFR_slope_dir', 'HbA1c_cov', 'HbA1c_ever_abnormal', 'HbA1c_frac_abnormal', 'HbA1c_recency_days', 'HbA1c_sd', 'HbA1c_slope', 'HbA1c_slope_dir', 'HCO3_ever_abnormal', 'HCO3_frac_abnormal', 'HCO3_recency_days', 'HDL_ever_abnormal', 'HDL_frac_abnormal', 'HDL_recency_days', 'HR_ever_abnormal', 'HR_frac_abnormal', 'HR_max', 'HR_mean', 'HR_min', 'HR_n', 'HR_recency_days', 'HR_sd', 'Hgb_ever_abnormal', 'Hgb_frac_abnormal', 'Hgb_recency_days', 'K_ever_abnormal', 'K_frac_abnormal', 'K_recency_days', 'LDL_ever_abnormal', 'LDL_frac_abnormal', 'LDL_recency_days', 'Na_ever_abnormal', 'Na_frac_abnormal', 'Na_recency_days', 'RBG_cov', 'RBG_ever_abnormal', 'RBG_frac_abnormal', 'RBG_recency_days', 'RBG_sd', 'RBG_slope', 'RBG_slope_dir', 'SBP_cov', 'SBP_ever_abnormal', 'SBP_frac_abnormal', 'SBP_max', 'SBP_mean', 'SBP_min', 'SBP_n', 'SBP_recency_days', 'SBP_sd', 'SBP_slope', 'SBP_slope_dir', 'DBP_cov', 'DBP_ever_abnormal', 'DBP_frac_abnormal', 'DBP_max', 'DBP_mean', 'DBP_min', 'DBP_n', 'DBP_recency_days', 'DBP_sd', 'DBP_slope', 'DBP_slope_dir', 'Screatinine_cov', 'Screatinine_ever_abnormal', 'Screatinine_frac_abnormal', 'Screatinine_recency_days', 'Screatinine_sd', 'Screatinine_slope', 'Screatinine_slope_dir', 'TG_ever_abnormal', 'TG_frac_abnormal', 'TG_recency_days', 'TotalBilirubin_ever_abnormal', 'TotalBilirubin_frac_abnormal', 'TotalBilirubin_recency_days', 'TotalChol_ever_abnormal', 'TotalChol_frac_abnormal', 'TotalChol_recency_days', 'UACR_cov', 'UACR_ever_abnormal', 'UACR_frac_abnormal', 'UACR_recency_days', 'UACR_sd', 'UACR_slope', 'UACR_slope_dir', 'UrAlb_cov', 'UrAlb_ever_abnormal', 'UrAlb_frac_abnormal', 'UrAlb_sd', 'UrAlb_slope_dir', 'UricAcid_ever_abnormal', 'UricAcid_frac_abnormal', 'UricAcid_recency_days', 'HR_last', 'SBP_last', 'DBP_last', 'Height_last', 'Weight_last', 'Weight_max', 'Weight_mean', 'Weight_min', 'Weight_n', 'Weight_recency_days', 'Weight_slope', 'Weight_slope_dir', 'dpp4i_ever', 'dpp4i_recent_30d', 'glp1ra_ever', 'glp1ra_recent_30d', 'insulin_ever', 'insulin_recent_30d', 'metformin_ever', 'metformin_recent_30d', 'sglt2i_ever', 'sglt2i_recent_30d', 'sulfonylureas_ever', 'sulfonylureas_recent_30d', 'thiazolidinediones_ever', 'thiazolidinediones_recent_30d', 'htn_acei_ever', 'htn_acei_recent_30d', 'htn_arb_ever', 'htn_arb_recent_30d', 'htn_beta_blockers_ever', 'htn_beta_blockers_recent_30d', 'htn_ccb_dhp_ever', 'htn_ccb_dhp_recent_30d', 'htn_ccb_non_dhp_ever', 'htn_ccb_non_dhp_recent_30d', 'htn_loops_ever', 'htn_loops_recent_30d', 'htn_mra_ever', 'htn_mra_recent_30d', 'htn_potassium_sparing_ever', 'htn_potassium_sparing_recent_30d', 'htn_thiazides_ever', 'htn_thiazides_recent_30d', 'lipid_bile_acid_seq_ever', 'lipid_bile_acid_seq_recent_30d', 'lipid_ezetimibe_ever', 'lipid_ezetimibe_recent_30d', 'lipid_fibrates_ever', 'lipid_fibrates_recent_30d', 'lipid_niacin_ever', 'lipid_niacin_recent_30d', 'lipid_statins_ever', 'lipid_statins_recent_30d', 'acute_kidney_injury', 'anxiety_disorder', 'atrial_fibrillation', 'chronic_hepatitis_c', 'chronic_liver_disease', 'cirrhosis', 'copd', 'coronary_artery_disease', 'diabetic_foot', 'diabetic_neuropathy', 'diabetic_retinopathy', 'gout', 'heart_failure', 'hiv', 'hypertension', 'major_depressive_disorder', 'metabolic_syndrome', 'myocardial_infarction', 'nephrolithiasis', 'obesity', 'peripheral_arterial_disease', 'psoriasis', 'psychotic_disorder', 'rheumatoid_arthritis', 'sle', 'sleep_apnea', 'steatotic_liver_disease', 'uti']

CANDIDATE_BIOME_PREDICTORS = [
    c for c in CANDIDATE_BIOME_MUTUAL_FEATURES
    if c not in {ID_COL, Y_COL}
]

if len(CANDIDATE_BIOME_MUTUAL_FEATURES) != 342:
    raise ValueError(
        f"Expected 342 entries from the shared BioMe script; "
        f"found {len(CANDIDATE_BIOME_MUTUAL_FEATURES)}."
    )

if len(CANDIDATE_BIOME_PREDICTORS) != 340:
    raise ValueError(
        f"Expected 340 candidate predictors; found {len(CANDIDATE_BIOME_PREDICTORS)}."
    )

print(f"Candidate BioMe mutual list: {len(CANDIDATE_BIOME_MUTUAL_FEATURES)} entries")
print(f"Candidate predictors: {len(CANDIDATE_BIOME_PREDICTORS)}")

## Load corrected AoU cohorts, split manifest, and corrected feature manifest

In [ ]:
FINAL_MODEL_PATHS = {
    landmark: (
        f"{COHORT_DIR}/DKD_Risk_FinalModel_Data_Landmark_"
        f"{landmark}_{SOURCE_DATE}.pkl"
    )
    for landmark in ["30d", "6m", "1y", "2y"]
}

FEATURE_MANIFEST_PATH = (
    f"{PRIMARY_RESULT_DIR}/final_model_feature_manifest_{SOURCE_DATE}.csv"
)
SPLIT_MANIFEST_PATH = (
    f"{PRIMARY_RESULT_DIR}/final_model_split_manifest_{SOURCE_DATE}.csv"
)

required_paths = list(FINAL_MODEL_PATHS.values()) + [
    FEATURE_MANIFEST_PATH,
    SPLIT_MANIFEST_PATH,
]
for path in required_paths:
    if not fs.exists(path):
        raise FileNotFoundError(path)

landmark_data = {
    landmark: read_gcs_pickle(path)
    for landmark, path in FINAL_MODEL_PATHS.items()
}
feature_manifest = read_gcs_csv(FEATURE_MANIFEST_PATH)
split_manifest = read_gcs_csv(SPLIT_MANIFEST_PATH)

for landmark, df in landmark_data.items():
    df[ID_COL] = pd.to_numeric(df[ID_COL], errors="raise").astype("int64")
    df[Y_COL] = pd.to_numeric(df[Y_COL], errors="raise").astype(int)
    if df[ID_COL].duplicated().any():
        raise ValueError(f"{landmark}: duplicate person_id in corrected AoU dataset")

split_manifest[ID_COL] = pd.to_numeric(
    split_manifest[ID_COL], errors="raise"
).astype("int64")
split_manifest[Y_COL] = pd.to_numeric(
    split_manifest[Y_COL], errors="raise"
).astype(int)

print("Corrected AoU cohorts loaded")
for landmark, df in landmark_data.items():
    print(
        f"  {landmark:>3}: N={len(df):,} | "
        f"cases={int(df[Y_COL].sum()):,} | controls={int((df[Y_COL]==0).sum()):,}"
    )

## Derive the final BioMe-compatible feature manifests

For each model, feature order is inherited from the corrected AoU feature manifest. A predictor is retained only if it is also present in the candidate BioMe mutual list.

In [ ]:
biome_features = {}
feature_audit_rows = []
feature_manifest_rows = []

for landmark, model_ids in PRIMARY_EXTERNAL_MODEL_ORDER.items():
    biome_features[landmark] = {}

    for model_id in model_ids:
        primary = (
            feature_manifest.loc[
                feature_manifest["landmark"].eq(landmark)
                & feature_manifest["model_id"].eq(model_id)
            ]
            .sort_values("feature_order")
            .reset_index(drop=True)
        )
        primary_features = primary["feature"].tolist()
        if not primary_features:
            raise ValueError(f"{landmark}/{model_id}: no corrected AoU feature manifest")

        mutual_features = [
            f for f in primary_features
            if f in CANDIDATE_BIOME_PREDICTORS
        ]
        dropped = [
            f for f in primary_features
            if f not in CANDIDATE_BIOME_PREDICTORS
        ]

        if not mutual_features:
            raise ValueError(f"{landmark}/{model_id}: mutual feature set is empty")

        missing_from_data = [
            f for f in mutual_features
            if f not in landmark_data[landmark].columns
        ]
        if missing_from_data:
            raise ValueError(
                f"{landmark}/{model_id}: corrected AoU dataset missing mutual features: "
                f"{missing_from_data}"
            )

        biome_features[landmark][model_id] = mutual_features

        for order, feature in enumerate(mutual_features):
            feature_manifest_rows.append({
                "landmark": landmark,
                "model_id": model_id,
                "feature_order": order,
                "feature": feature,
            })

        feature_audit_rows.append({
            "landmark": landmark,
            "model_id": model_id,
            "corrected_aou_n_features": len(primary_features),
            "biome_compatible_n_features": len(mutual_features),
            "n_dropped_not_in_candidate_mutual": len(dropped),
            "dropped_not_in_candidate_mutual": " | ".join(dropped),
        })

# Nesting must survive intersection.
for landmark, model_ids in PRIMARY_EXTERNAL_MODEL_ORDER.items():
    previous = set()
    for model_id in model_ids:
        current = set(biome_features[landmark][model_id])
        if not previous.issubset(current):
            raise ValueError(f"{landmark}: mutual feature layers are not nested at {model_id}")
        previous = current

biome_feature_manifest = pd.DataFrame(feature_manifest_rows)
biome_feature_audit = pd.DataFrame(feature_audit_rows)

FEATURE_OUT = f"{EXTERNAL_RESULT_DIR}/aou_biome_feature_manifest_{RUN_DATE}.csv"
FEATURE_AUDIT_OUT = f"{EXTERNAL_RESULT_DIR}/aou_biome_feature_audit_{RUN_DATE}.csv"
CANDIDATE_OUT = f"{EXTERNAL_RESULT_DIR}/biome_candidate_mutual_features_{RUN_DATE}.csv"

save_gcs_csv(biome_feature_manifest, FEATURE_OUT)
save_gcs_csv(biome_feature_audit, FEATURE_AUDIT_OUT)
save_gcs_csv(
    pd.DataFrame({"feature": CANDIDATE_BIOME_MUTUAL_FEATURES}),
    CANDIDATE_OUT,
)

display(biome_feature_audit)

### Required BioMe-side check before the collaborator run

The revised BioMe extraction script must confirm that **every predictor in `aou_biome_feature_manifest_20260823.csv` is actually produced with the same definition and timing**. If any required feature cannot be reproduced reliably, stop before sending the package, remove that feature from the compatibility inventory, and rerun this notebook in AoU. Once the package is sent, the scoring script should fail rather than silently substitute a missing predictor.

## Validate and reconstruct the frozen corrected AoU split

In [ ]:
split_lookup = {}
cohort_order = {}

for landmark in PRIMARY_EXTERNAL_MODEL_ORDER:
    x = split_manifest.loc[split_manifest["landmark"].eq(landmark)].copy()

    if x[ID_COL].duplicated().any():
        raise ValueError(f"{landmark}: duplicate person_id in split manifest")

    data_ids = set(landmark_data[landmark][ID_COL].astype(int))
    split_ids = set(x[ID_COL].astype(int))
    if data_ids != split_ids:
        raise ValueError(f"{landmark}: corrected dataset and split manifest differ")

    split_lookup[landmark] = dict(zip(x[ID_COL].astype(int), x["split"]))
    cohort_order[landmark] = dict(zip(x[ID_COL].astype(int), x["cohort_order"].astype(int)))

    tab = (
        x.groupby("split")[Y_COL]
        .agg(N="size", cases="sum")
        .reindex(["train", "cal", "test"])
    )
    tab["controls"] = tab["N"] - tab["cases"]
    print(f"\n{landmark}")
    display(tab)

# 30d and 6m are intentionally identical in participants/outcomes/splits.
a = (
    split_manifest.loc[split_manifest["landmark"].eq("30d"), [ID_COL, Y_COL, "split"]]
    .sort_values(ID_COL).reset_index(drop=True)
)
b = (
    split_manifest.loc[split_manifest["landmark"].eq("6m"), [ID_COL, Y_COL, "split"]]
    .sort_values(ID_COL).reset_index(drop=True)
)
if not a.equals(b):
    raise ValueError("30d and 6m split identity check failed")
print("\n30d/6m participant, outcome, and split identity: PASSED")

In [ ]:
def get_partition(df, landmark, split_name):
    ids = {
        pid for pid, split_name_i in split_lookup[landmark].items()
        if split_name_i == split_name
    }
    out = df.loc[df[ID_COL].astype(int).isin(ids)].copy()
    out["_cohort_order"] = out[ID_COL].astype(int).map(cohort_order[landmark])
    return (
        out.sort_values("_cohort_order")
        .drop(columns="_cohort_order")
        .reset_index(drop=True)
    )


def make_model_df(landmark, model_id):
    features = biome_features[landmark][model_id]
    return landmark_data[landmark][[ID_COL, Y_COL] + features].copy()


def prepare_xy(df, features):
    X = df[features].astype(np.float32)
    y = df[Y_COL].astype(int).to_numpy()
    return X, y

## Train BioMe-compatible primary XGBoost models in AoU

Each model is tuned independently on AoU TRAIN using the same 20-trial search space as the corrected primary analysis, then calibrated on AoU CAL.

In [ ]:
def train_calibrate_mutual_xgb(landmark, model_id):
    features = biome_features[landmark][model_id]
    df = make_model_df(landmark, model_id)

    train_df = get_partition(df, landmark, "train")
    cal_df = get_partition(df, landmark, "cal")
    test_df = get_partition(df, landmark, "test")

    X_train, y_train = prepare_xy(train_df, features)
    X_cal, y_cal = prepare_xy(cal_df, features)
    X_test, y_test = prepare_xy(test_df, features)

    if list(X_train.columns) != features:
        raise ValueError(f"{landmark}/{model_id}: feature order changed")

    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

    def objective(trial):
        params = {
            "objective": "binary:logistic",
            "eval_metric": "auc",
            "tree_method": "hist",
            "gamma": trial.suggest_float("gamma", *XGB_BOUNDS["gamma"], log=True),
            "max_depth": trial.suggest_int("max_depth", *XGB_BOUNDS["max_depth"]),
            "min_child_weight": trial.suggest_int(
                "min_child_weight", *XGB_BOUNDS["min_child_weight"]
            ),
            "subsample": trial.suggest_float("subsample", *XGB_BOUNDS["subsample"]),
            "colsample_bytree": trial.suggest_float(
                "colsample_bytree", *XGB_BOUNDS["colsample_bytree"]
            ),
            "learning_rate": trial.suggest_float(
                "learning_rate", *XGB_BOUNDS["learning_rate"]
            ),
            "n_estimators": trial.suggest_int(
                "n_estimators", *XGB_BOUNDS["n_estimators"]
            ),
            "reg_alpha": trial.suggest_float(
                "reg_alpha", *XGB_BOUNDS["reg_alpha"], log=True
            ),
            "reg_lambda": trial.suggest_float(
                "reg_lambda", *XGB_BOUNDS["reg_lambda"], log=True
            ),
            "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 2.0),
            "n_jobs": XGB_N_JOBS,
            "random_state": SEED,
        }
        model = xgb.XGBClassifier(**params)
        scores = cross_val_score(
            model, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=1
        )
        return float(scores.mean())

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
    )
    study.optimize(objective, n_trials=N_XGB_TRIALS)

    selected = dict(study.best_params)
    fit_params = {
        **selected,
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "tree_method": "hist",
        "n_jobs": XGB_N_JOBS,
        "random_state": SEED,
    }
    model = xgb.XGBClassifier(**fit_params)
    model.fit(X_train, y_train)

    fitted_features = list(model.get_booster().feature_names)
    if fitted_features != features:
        raise ValueError(f"{landmark}/{model_id}: fitted feature order changed")

    calibrator = CalibratedClassifierCV(model, method="isotonic", cv="prefit")
    calibrator.fit(X_cal, y_cal)

    stem = f"dkd_biome_{landmark}_{model_id}_xgb"
    model_path = f"{EXTERNAL_MODEL_DIR}/{stem}_raw_{RUN_DATE}.pkl"
    cal_path = f"{EXTERNAL_MODEL_DIR}/{stem}_isotonic_calibrator_{RUN_DATE}.pkl"
    pred_path = f"{EXTERNAL_RESULT_DIR}/{stem}_predictions_{RUN_DATE}.csv"

    save_gcs_pickle(model, model_path)
    save_gcs_pickle(calibrator, cal_path)

    pred_frames = []
    for split_name, part, X, y in [
        ("cal", cal_df, X_cal, y_cal),
        ("test", test_df, X_test, y_test),
    ]:
        pred_frames.append(pd.DataFrame({
            ID_COL: part[ID_COL].astype(int).to_numpy(),
            "split": split_name,
            Y_COL: y,
            "p_raw": model.predict_proba(X)[:, 1],
            "p_calibrated": calibrator.predict_proba(X)[:, 1],
        }))

    preds = pd.concat(pred_frames, ignore_index=True)
    save_gcs_csv(preds, pred_path)

    return {
        "landmark": landmark,
        "model_id": model_id,
        "analysis_group": "primary_biome_compatible",
        "algorithm": "XGB",
        "layer": model_id,
        "features": features,
        "n_features": len(features),
        "best_cv_auroc": float(study.best_value),
        "best_params": selected,
        "model_path": model_path,
        "calibrator_path": cal_path,
        "prediction_path": pred_path,
        "predictions": preds,
    }

In [ ]:
primary_results = {}

for landmark, model_ids in PRIMARY_EXTERNAL_MODEL_ORDER.items():
    primary_results[landmark] = {}
    for model_id in model_ids:
        print("\n" + "=" * 90)
        print(
            f"Training {landmark} {model_id} | "
            f"BioMe-compatible features={len(biome_features[landmark][model_id])}"
        )
        result = train_calibrate_mutual_xgb(landmark, model_id)
        primary_results[landmark][model_id] = result
        print(f"Best TRAIN CV AUROC: {result['best_cv_auroc']:.5f}")

## One-year secondary models for the external-validation package

These models are prepared now so the collaborator can score them in the same run if needed:

1. **Clinical LR:** age at first documented T2D date, eGFR, UACR, and HbA1c.
2. **Mutual M4 LR:** logistic regression using the complete one-year BioMe-compatible M4 feature set.
3. **Mutual M4 RF:** random forest using the same feature set.

All are trained on AoU TRAIN, isotonic-calibrated on AoU CAL, and evaluated on AoU TEST. They remain secondary analyses.

In [ ]:
def make_lr_pipeline(C=1.0, *, add_indicator=True):
    return Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median", add_indicator=add_indicator),
        ),
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                C=float(C), penalty="l2", solver="lbfgs",
                max_iter=5000, random_state=SEED,
            ),
        ),
    ])


def make_rf_pipeline(params, *, n_jobs):
    return Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median", add_indicator=True),
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=int(params["n_estimators"]),
                max_depth=int(params["max_depth"]),
                min_samples_leaf=int(params["min_samples_leaf"]),
                max_features=float(params["max_features"]),
                n_jobs=n_jobs,
                random_state=SEED,
            ),
        ),
    ])


def fit_calibrate_secondary(
    *, model, features, model_id, algorithm, best_cv_auroc, best_params,
    add_label, landmark="1y"
):
    base = landmark_data[landmark][[ID_COL, Y_COL] + features].copy()
    train_df = get_partition(base, landmark, "train")
    cal_df = get_partition(base, landmark, "cal")
    test_df = get_partition(base, landmark, "test")

    X_train, y_train = prepare_xy(train_df, features)
    X_cal, y_cal = prepare_xy(cal_df, features)
    X_test, y_test = prepare_xy(test_df, features)

    model.fit(X_train, y_train)
    calibrator = CalibratedClassifierCV(model, method="isotonic", cv="prefit")
    calibrator.fit(X_cal, y_cal)

    stem = f"dkd_biome_{landmark}_{model_id}_{algorithm.lower()}"
    model_path = f"{EXTERNAL_MODEL_DIR}/{stem}_raw_{RUN_DATE}.pkl"
    cal_path = f"{EXTERNAL_MODEL_DIR}/{stem}_isotonic_calibrator_{RUN_DATE}.pkl"
    pred_path = f"{EXTERNAL_RESULT_DIR}/{stem}_predictions_{RUN_DATE}.csv"

    save_gcs_pickle(model, model_path)
    save_gcs_pickle(calibrator, cal_path)

    pred_frames = []
    for split_name, part, X, y in [
        ("cal", cal_df, X_cal, y_cal),
        ("test", test_df, X_test, y_test),
    ]:
        pred_frames.append(pd.DataFrame({
            ID_COL: part[ID_COL].astype(int).to_numpy(),
            "split": split_name,
            Y_COL: y,
            "p_raw": model.predict_proba(X)[:, 1],
            "p_calibrated": calibrator.predict_proba(X)[:, 1],
        }))

    preds = pd.concat(pred_frames, ignore_index=True)
    save_gcs_csv(preds, pred_path)

    return {
        "landmark": landmark,
        "model_id": model_id,
        "analysis_group": add_label,
        "algorithm": algorithm,
        "layer": "M4" if model_id != "clinical_lr" else "clinical_4var",
        "features": features,
        "n_features": len(features),
        "best_cv_auroc": float(best_cv_auroc),
        "best_params": best_params,
        "model_path": model_path,
        "calibrator_path": cal_path,
        "prediction_path": pred_path,
        "predictions": preds,
    }

In [ ]:
secondary_results = {}

if RUN_1Y_SECONDARY_MODELS:
    m4_features = biome_features["1y"]["M4"]
    train_1y = get_partition(
        landmark_data["1y"][[ID_COL, Y_COL] + m4_features],
        "1y",
        "train",
    )
    y_train_1y = train_1y[Y_COL].to_numpy()
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

    # Clinical four-variable comparator.
    clinical_features = ["age_at_t2d", "eGFR_last", "UACR_last", "HbA1c_last"]
    missing_clinical = [f for f in clinical_features if f not in m4_features]
    if missing_clinical:
        raise ValueError(
            f"Clinical comparator feature(s) absent from mutual M4: {missing_clinical}"
        )

    def clinical_lr_objective(trial):
        C = trial.suggest_float("C", 1e-4, 1e4, log=True)
        model = make_lr_pipeline(C=C, add_indicator=False)
        scores = cross_val_score(
            model,
            train_1y[clinical_features],
            y_train_1y,
            cv=cv,
            scoring="roc_auc",
            n_jobs=CV_N_JOBS,
        )
        return float(scores.mean())

    study = optuna.create_study(
        direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED)
    )
    study.optimize(clinical_lr_objective, n_trials=N_LR_TRIALS)
    best = dict(study.best_params)
    secondary_results["clinical_lr"] = fit_calibrate_secondary(
        model=make_lr_pipeline(C=best["C"], add_indicator=False),
        features=clinical_features,
        model_id="clinical_lr",
        algorithm="LR",
        best_cv_auroc=study.best_value,
        best_params=best,
        add_label="clinical_benchmark",
    )

    # Logistic regression on mutual M4.
    def m4_lr_objective(trial):
        C = trial.suggest_float("C", 1e-4, 1e4, log=True)
        model = make_lr_pipeline(C=C, add_indicator=True)
        scores = cross_val_score(
            model,
            train_1y[m4_features],
            y_train_1y,
            cv=cv,
            scoring="roc_auc",
            n_jobs=CV_N_JOBS,
        )
        return float(scores.mean())

    study = optuna.create_study(
        direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED)
    )
    study.optimize(m4_lr_objective, n_trials=N_LR_TRIALS)
    best = dict(study.best_params)
    secondary_results["M4_LR"] = fit_calibrate_secondary(
        model=make_lr_pipeline(C=best["C"], add_indicator=True),
        features=m4_features,
        model_id="M4",
        algorithm="LR",
        best_cv_auroc=study.best_value,
        best_params=best,
        add_label="algorithmic_sensitivity",
    )

    # Random forest on mutual M4.
    def m4_rf_objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 300, 900, step=100),
            "max_depth": trial.suggest_int("max_depth", 6, 30),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
            "max_features": trial.suggest_float("max_features", 0.20, 1.00),
        }
        model = make_rf_pipeline(params, n_jobs=RF_CV_N_JOBS)
        scores = cross_val_score(
            model,
            train_1y[m4_features],
            y_train_1y,
            cv=cv,
            scoring="roc_auc",
            n_jobs=CV_N_JOBS,
        )
        return float(scores.mean())

    study = optuna.create_study(
        direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED)
    )
    study.optimize(m4_rf_objective, n_trials=N_RF_TRIALS)
    best = dict(study.best_params)
    secondary_results["M4_RF"] = fit_calibrate_secondary(
        model=make_rf_pipeline(best, n_jobs=RF_FINAL_N_JOBS),
        features=m4_features,
        model_id="M4",
        algorithm="RF",
        best_cv_auroc=study.best_value,
        best_params=best,
        add_label="algorithmic_sensitivity",
    )

    print("One-year secondary models trained:", list(secondary_results))
else:
    print("One-year secondary model training disabled by configuration.")

## Evaluation helpers

The primary probability scale is the isotonic-calibrated probability. Fixed-policy thresholds are learned on AoU CAL and then evaluated unchanged on AoU TEST.

In [ ]:
def safe_auroc(y, p):
    y = np.asarray(y)
    p = np.asarray(p)
    return np.nan if np.unique(y).size < 2 else float(roc_auc_score(y, p))


def safe_auprc(y, p):
    y = np.asarray(y)
    p = np.asarray(p)
    return np.nan if np.unique(y).size < 2 else float(average_precision_score(y, p))


def confusion_metrics(y, p, threshold):
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)
    pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
        "Sensitivity": tp / (tp + fn) if (tp + fn) else np.nan,
        "Specificity": tn / (tn + fp) if (tn + fp) else np.nan,
        "PPV": tp / (tp + fp) if (tp + fp) else np.nan,
        "NPV": tn / (tn + fn) if (tn + fn) else np.nan,
        "Accuracy": (tp + tn) / len(y) if len(y) else np.nan,
    }


def choose_threshold_for_specificity(y, p, target_specificity):
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)
    fpr, tpr, thresholds = roc_curve(y, p)
    specificity = 1.0 - fpr
    eligible = np.where(specificity >= target_specificity)[0]
    if len(eligible) == 0:
        raise RuntimeError("No CAL threshold achieves target specificity")

    max_tpr = np.max(tpr[eligible])
    candidates = eligible[np.isclose(tpr[eligible], max_tpr, rtol=0, atol=1e-12)]
    chosen_idx = candidates[np.argmin(thresholds[candidates])]
    return {
        "threshold": float(thresholds[chosen_idx]),
        "cal_sensitivity": float(tpr[chosen_idx]),
        "cal_specificity": float(specificity[chosen_idx]),
    }


def oe_ratio(y, p):
    observed = float(np.sum(y))
    expected = float(np.sum(p))
    return np.nan if expected <= 0 else observed / expected


def calibration_intercept_slope(y, p):
    y = np.asarray(y).astype(float)
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    z = logit(p)
    X = np.column_stack([np.ones(len(z)), z])
    beta = np.array([0.0, 1.0], dtype=float)
    for _ in range(100):
        eta = X @ beta
        mu = expit(eta)
        w = np.clip(mu * (1 - mu), 1e-9, None)
        score = X.T @ (y - mu)
        hessian = -(X.T * w) @ X
        step = np.linalg.solve(hessian, score)
        beta_new = beta - step
        if np.max(np.abs(beta_new - beta)) < 1e-10:
            beta = beta_new
            break
        beta = beta_new
    return float(beta[0]), float(beta[1])


def calibration_in_the_large(y, p):
    y = np.asarray(y).astype(float)
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    z = logit(p)
    target = float(np.sum(y))
    def score(alpha):
        return float(np.sum(expit(z + alpha)) - target)
    return float(brentq(score, -30.0, 30.0))


def stratified_bootstrap_indices(y, rng):
    y = np.asarray(y).astype(int)
    idx0 = np.where(y == 0)[0]
    idx1 = np.where(y == 1)[0]
    out = np.concatenate([
        rng.choice(idx0, size=len(idx0), replace=True),
        rng.choice(idx1, size=len(idx1), replace=True),
    ])
    rng.shuffle(out)
    return out


def percentile_ci(values, alpha=0.05):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    return (
        float(np.quantile(values, alpha / 2)),
        float(np.quantile(values, 1 - alpha / 2)),
    )


def bootstrap_test_metrics(y, p, threshold, n_boot=N_BOOT, seed=BOOT_SEED):
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)
    rng = np.random.default_rng(seed)
    samples = {
        "AUROC": [], "AUPRC": [], "Brier": [], "O_E": [],
        "Calibration_intercept": [], "Calibration_slope": [], "CITL": [],
        "Sensitivity": [], "Specificity": [], "PPV": [], "NPV": [],
    }
    for _ in range(n_boot):
        idx = stratified_bootstrap_indices(y, rng)
        yb, pb = y[idx], p[idx]
        samples["AUROC"].append(safe_auroc(yb, pb))
        samples["AUPRC"].append(safe_auprc(yb, pb))
        samples["Brier"].append(brier_score_loss(yb, pb))
        samples["O_E"].append(oe_ratio(yb, pb))
        try:
            intercept, slope = calibration_intercept_slope(yb, pb)
        except Exception:
            intercept, slope = np.nan, np.nan
        samples["Calibration_intercept"].append(intercept)
        samples["Calibration_slope"].append(slope)
        try:
            citl = calibration_in_the_large(yb, pb)
        except Exception:
            citl = np.nan
        samples["CITL"].append(citl)
        cm = confusion_metrics(yb, pb, threshold)
        for metric in ["Sensitivity", "Specificity", "PPV", "NPV"]:
            samples[metric].append(cm[metric])
    return {metric: percentile_ci(values) for metric, values in samples.items()}

## Collect all frozen AoU prediction objects

In [ ]:
all_results = []

for landmark, models in primary_results.items():
    all_results.extend(models.values())
all_results.extend(secondary_results.values())

if not all_results:
    raise RuntimeError("No models available for evaluation")

print(f"Models available for evaluation: {len(all_results)}")
for r in all_results:
    print(
        f"  {r['landmark']:>3} | {r['model_id']:<12} | {r['algorithm']:<3} | "
        f"features={r['n_features']:>3} | group={r['analysis_group']}"
    )

## Learn AoU CAL thresholds at 80%, 85%, 90%, and 95% target specificity

Because isotonic calibration yields discrete probability values, several target specificity levels can occasionally select the same threshold. The table therefore records the **achieved CAL specificity** as well as the target.

In [ ]:
threshold_rows = []

for r in all_results:
    cal = r["predictions"].loc[r["predictions"]["split"].eq("cal")].copy()
    for target_spec in SPEC_LEVELS:
        info = choose_threshold_for_specificity(
            cal[Y_COL].to_numpy(),
            cal["p_calibrated"].to_numpy(),
            target_specificity=target_spec,
        )
        threshold_rows.append({
            "analysis_group": r["analysis_group"],
            "landmark": r["landmark"],
            "model_id": r["model_id"],
            "algorithm": r["algorithm"],
            "target_specificity": target_spec,
            "threshold_cal": info["threshold"],
            "cal_specificity": info["cal_specificity"],
            "cal_sensitivity": info["cal_sensitivity"],
            "cal_n": len(cal),
            "cal_cases": int(cal[Y_COL].sum()),
        })

threshold_df = pd.DataFrame(threshold_rows)
THRESHOLD_OUT = f"{EXTERNAL_RESULT_DIR}/aou_biome_cal_thresholds_{RUN_DATE}.csv"
save_gcs_csv(threshold_df, THRESHOLD_OUT)
display(threshold_df)

## AoU TEST performance at the primary 90%-specificity policy

Bootstrap confidence intervals use 2,000 stratified replicates, matching the corrected AoU performance notebook.

In [ ]:
performance_rows = []

for r in all_results:
    test = r["predictions"].loc[r["predictions"]["split"].eq("test")].copy()
    y = test[Y_COL].to_numpy()
    p = test["p_calibrated"].to_numpy()

    threshold = float(
        threshold_df.loc[
            threshold_df["landmark"].eq(r["landmark"])
            & threshold_df["model_id"].eq(r["model_id"])
            & threshold_df["algorithm"].eq(r["algorithm"])
            & threshold_df["target_specificity"].eq(PRIMARY_SPEC),
            "threshold_cal",
        ].iloc[0]
    )

    cm = confusion_metrics(y, p, threshold)
    intercept, slope = calibration_intercept_slope(y, p)
    point = {
        "AUROC": safe_auroc(y, p),
        "AUPRC": safe_auprc(y, p),
        "Brier": float(brier_score_loss(y, p)),
        "O_E": oe_ratio(y, p),
        "Calibration_intercept": intercept,
        "Calibration_slope": slope,
        "CITL": calibration_in_the_large(y, p),
        **cm,
    }
    ci = bootstrap_test_metrics(y, p, threshold)

    row = {
        "analysis_group": r["analysis_group"],
        "landmark": r["landmark"],
        "model_id": r["model_id"],
        "algorithm": r["algorithm"],
        "n_features": r["n_features"],
        "best_cv_auroc": r["best_cv_auroc"],
        "n_test": len(test),
        "n_cases": int(y.sum()),
        "n_controls": int((y == 0).sum()),
        "prevalence": float(y.mean()),
        "threshold_cal": threshold,
    }
    for metric, value in point.items():
        row[metric] = value
        if metric in ci:
            row[f"{metric}_ci_low"] = ci[metric][0]
            row[f"{metric}_ci_high"] = ci[metric][1]
    performance_rows.append(row)

performance_df = pd.DataFrame(performance_rows)
PERFORMANCE_OUT = f"{EXTERNAL_RESULT_DIR}/aou_biome_test_performance_{RUN_DATE}.csv"
save_gcs_csv(performance_df, PERFORMANCE_OUT)

## AoU TEST performance across all four fixed-specificity policies

In [ ]:
policy_rows = []

for r in all_results:
    test = r["predictions"].loc[r["predictions"]["split"].eq("test")].copy()
    y = test[Y_COL].to_numpy()
    p = test["p_calibrated"].to_numpy()

    for target_spec in SPEC_LEVELS:
        trow = threshold_df.loc[
            threshold_df["landmark"].eq(r["landmark"])
            & threshold_df["model_id"].eq(r["model_id"])
            & threshold_df["algorithm"].eq(r["algorithm"])
            & threshold_df["target_specificity"].eq(target_spec)
        ].iloc[0]
        cm = confusion_metrics(y, p, float(trow["threshold_cal"]))
        policy_rows.append({
            "analysis_group": r["analysis_group"],
            "landmark": r["landmark"],
            "model_id": r["model_id"],
            "algorithm": r["algorithm"],
            "target_specificity": target_spec,
            "threshold_cal": float(trow["threshold_cal"]),
            "cal_specificity": float(trow["cal_specificity"]),
            "cal_sensitivity": float(trow["cal_sensitivity"]),
            "n_test": len(test),
            "n_cases": int(y.sum()),
            **cm,
        })

policy_df = pd.DataFrame(policy_rows)
POLICY_OUT = f"{EXTERNAL_RESULT_DIR}/aou_biome_policy_80_85_90_95_{RUN_DATE}.csv"
save_gcs_csv(policy_df, POLICY_OUT)

## Raw-versus-calibrated discrimination audit

This table confirms the behavior of the isotonic calibration step. Calibration can alter ranking slightly because isotonic probabilities are discrete, so both raw and calibrated discrimination are retained for auditability.

In [ ]:
raw_cal_rows = []

for r in all_results:
    test = r["predictions"].loc[r["predictions"]["split"].eq("test")].copy()
    y = test[Y_COL].to_numpy()
    raw_cal_rows.append({
        "analysis_group": r["analysis_group"],
        "landmark": r["landmark"],
        "model_id": r["model_id"],
        "algorithm": r["algorithm"],
        "AUROC_raw": safe_auroc(y, test["p_raw"].to_numpy()),
        "AUROC_calibrated": safe_auroc(y, test["p_calibrated"].to_numpy()),
        "AUPRC_raw": safe_auprc(y, test["p_raw"].to_numpy()),
        "AUPRC_calibrated": safe_auprc(y, test["p_calibrated"].to_numpy()),
        "Brier_raw": float(brier_score_loss(y, test["p_raw"].to_numpy())),
        "Brier_calibrated": float(brier_score_loss(y, test["p_calibrated"].to_numpy())),
    })

raw_cal_df = pd.DataFrame(raw_cal_rows)
RAW_CAL_OUT = f"{EXTERNAL_RESULT_DIR}/aou_biome_raw_vs_calibrated_{RUN_DATE}.csv"
save_gcs_csv(raw_cal_df, RAW_CAL_OUT)

## Compare BioMe-compatible AoU models with the full corrected AoU models

This is an internal AoU check of the performance cost, if any, from restricting each model to predictors that can be reproduced in BioMe. It is **not** part of the external validation itself.

In [ ]:
PRIMARY_PERFORMANCE_PATH = (
    f"{PRIMARY_RESULT_DIR}/test_performance_all_models_{SOURCE_DATE}.csv"
)

if fs.exists(PRIMARY_PERFORMANCE_PATH):
    full_perf = read_gcs_csv(PRIMARY_PERFORMANCE_PATH)
    mutual_primary = performance_df.loc[
        performance_df["analysis_group"].eq("primary_biome_compatible")
    ].copy()

    full_keep = full_perf[[
        "landmark", "model_id", "AUROC", "AUPRC", "Brier",
        "Sensitivity", "Specificity", "PPV", "NPV"
    ]].rename(columns={
        "AUROC": "full_AUROC",
        "AUPRC": "full_AUPRC",
        "Brier": "full_Brier",
        "Sensitivity": "full_Sensitivity_90",
        "Specificity": "full_Specificity_90",
        "PPV": "full_PPV_90",
        "NPV": "full_NPV_90",
    })

    compare_df = mutual_primary.merge(
        full_keep, on=["landmark", "model_id"], how="left"
    )
    compare_df["delta_AUROC_mutual_minus_full"] = (
        compare_df["AUROC"] - compare_df["full_AUROC"]
    )
    compare_df["delta_AUPRC_mutual_minus_full"] = (
        compare_df["AUPRC"] - compare_df["full_AUPRC"]
    )
    compare_df["delta_Brier_mutual_minus_full"] = (
        compare_df["Brier"] - compare_df["full_Brier"]
    )

    compare_out = compare_df[[
        "landmark", "model_id", "n_features",
        "full_AUROC", "AUROC", "delta_AUROC_mutual_minus_full",
        "full_AUPRC", "AUPRC", "delta_AUPRC_mutual_minus_full",
        "full_Brier", "Brier", "delta_Brier_mutual_minus_full",
    ]].copy()
    COMPARE_OUT = f"{EXTERNAL_RESULT_DIR}/aou_full_vs_biome_compatible_{RUN_DATE}.csv"
    save_gcs_csv(compare_out, COMPARE_OUT)
else:
    compare_out = pd.DataFrame()
    print("Primary corrected performance table not found; comparison skipped")

## Freeze the collaborator manifests

The final BioMe scoring package uses two machine-readable manifests:

- a **model-specific feature manifest** containing the exact ordered features for every frozen model, including the one-year secondary models; and
- a **model manifest** containing model/calibrator file names and all AoU CAL-derived thresholds.

The BioMe scoring script should read these files directly and **stop with an error if any required feature is absent**.

In [ ]:
# Final model-specific feature manifest for every object that may be scored in BioMe.
# Including algorithm and analysis_group avoids ambiguity for the secondary models.
collab_feature_rows = []
for r in all_results:
    for feature_order, feature in enumerate(r["features"]):
        collab_feature_rows.append({
            "analysis_group": r["analysis_group"],
            "landmark": r["landmark"],
            "model_id": r["model_id"],
            "algorithm": r["algorithm"],
            "feature_order": feature_order,
            "feature": feature,
        })

collaborator_feature_manifest = pd.DataFrame(collab_feature_rows)
COLLAB_FEATURE_OUT = (
    f"{EXTERNAL_RESULT_DIR}/aou_biome_model_feature_manifest_{RUN_DATE}.csv"
)
save_gcs_csv(collaborator_feature_manifest, COLLAB_FEATURE_OUT)

# Every model must have exactly n_features rows in the final manifest.
manifest_counts = (
    collaborator_feature_manifest
    .groupby(["analysis_group", "landmark", "model_id", "algorithm"])
    .size()
    .reset_index(name="manifest_n_features")
)
expected_counts = pd.DataFrame([
    {
        "analysis_group": r["analysis_group"],
        "landmark": r["landmark"],
        "model_id": r["model_id"],
        "algorithm": r["algorithm"],
        "expected_n_features": r["n_features"],
    }
    for r in all_results
])
manifest_check = expected_counts.merge(
    manifest_counts,
    on=["analysis_group", "landmark", "model_id", "algorithm"],
    how="left",
)
if not (manifest_check["expected_n_features"] == manifest_check["manifest_n_features"]).all():
    raise ValueError("Final collaborator feature manifest count check failed")

display(manifest_check)

In [ ]:
manifest_rows = []

for r in all_results:
    row = {
        "run_date": RUN_DATE,
        "landmark": r["landmark"],
        "model_id": r["model_id"],
        "algorithm": r["algorithm"],
        "analysis_group": r["analysis_group"],
        "layer": r["layer"],
        "n_features": r["n_features"],
        "best_train_cv_auroc": r["best_cv_auroc"],
        "raw_model_file": basename(r["model_path"]),
        "calibrator_file": basename(r["calibrator_path"]),
        "prediction_file_aou": basename(r["prediction_path"]),
        "raw_model_gcs_path": r["model_path"],
        "calibrator_gcs_path": r["calibrator_path"],
        "feature_manifest_file": basename(COLLAB_FEATURE_OUT),
    }
    for target_spec in SPEC_LEVELS:
        trow = threshold_df.loc[
            threshold_df["landmark"].eq(r["landmark"])
            & threshold_df["model_id"].eq(r["model_id"])
            & threshold_df["algorithm"].eq(r["algorithm"])
            & threshold_df["target_specificity"].eq(target_spec)
        ].iloc[0]
        suffix = str(int(round(target_spec * 100)))
        row[f"threshold_spec_{suffix}"] = float(trow["threshold_cal"])
        row[f"achieved_cal_spec_{suffix}"] = float(trow["cal_specificity"])
    manifest_rows.append(row)

model_manifest = pd.DataFrame(manifest_rows)
MODEL_MANIFEST_OUT = f"{EXTERNAL_RESULT_DIR}/aou_biome_model_manifest_{RUN_DATE}.csv"
save_gcs_csv(model_manifest, MODEL_MANIFEST_OUT)

# Save selected hyperparameters separately for reproducibility.
hyper_rows = []
for r in all_results:
    hyper_rows.append({
        "landmark": r["landmark"],
        "model_id": r["model_id"],
        "algorithm": r["algorithm"],
        "analysis_group": r["analysis_group"],
        "n_features": r["n_features"],
        "best_cv_auroc": r["best_cv_auroc"],
        "best_params_json": json.dumps(r["best_params"], sort_keys=True),
    })
hyper_df = pd.DataFrame(hyper_rows)
HYPER_OUT = f"{EXTERNAL_RESULT_DIR}/aou_biome_selected_hyperparameters_{RUN_DATE}.csv"
save_gcs_csv(hyper_df, HYPER_OUT)

summary_json = {
    "run_date": RUN_DATE,
    "source_corrected_aou_date": SOURCE_DATE,
    "random_seed": SEED,
    "candidate_mutual_entries_in_old_python_script": len(CANDIDATE_BIOME_MUTUAL_FEATURES),
    "candidate_mutual_predictors": len(CANDIDATE_BIOME_PREDICTORS),
    "primary_models": {
        k: v for k, v in PRIMARY_EXTERNAL_MODEL_ORDER.items()
    },
    "secondary_1y_models_included": bool(RUN_1Y_SECONDARY_MODELS),
    "xgb_optuna_trials": N_XGB_TRIALS,
    "lr_optuna_trials": N_LR_TRIALS,
    "rf_optuna_trials": N_RF_TRIALS,
    "cv_folds": CV_FOLDS,
    "calibration": "isotonic on AoU CAL",
    "threshold_targets": SPEC_LEVELS,
    "primary_policy_target_specificity": PRIMARY_SPEC,
    "bootstrap_replicates": N_BOOT,
    "external_validation_rule": (
        "BioMe must use frozen AoU models, calibrators, exact feature order, "
        "and AoU CAL-derived thresholds without retraining or recalibration."
    ),
}
SUMMARY_OUT = f"{EXTERNAL_RESULT_DIR}/aou_biome_prep_summary_{RUN_DATE}.json"
save_gcs_json(summary_json, SUMMARY_OUT)

# Final AoU report — BioMe-compatible models

The tables below are the final internal AoU audit before the external-validation package is sent. No plots are produced.

In [ ]:
# Compact feature-count report.
print("=" * 100)
print("BIOme-COMPATIBLE FEATURE COUNTS")
print("=" * 100)
display(
    biome_feature_audit[[
        "landmark", "model_id", "corrected_aou_n_features",
        "biome_compatible_n_features", "n_dropped_not_in_candidate_mutual"
    ]]
)

# Primary AoU TEST performance at the 90% policy.
print("\n" + "=" * 100)
print("AoU TEST PERFORMANCE — PRIMARY BIOME-COMPATIBLE XGBOOST MODELS")
print("=" * 100)
primary_report = performance_df.loc[
    performance_df["analysis_group"].eq("primary_biome_compatible")
].copy()

display(primary_report[[
    "landmark", "model_id", "n_features", "n_test", "n_cases",
    "AUROC", "AUROC_ci_low", "AUROC_ci_high",
    "AUPRC", "AUPRC_ci_low", "AUPRC_ci_high",
    "Brier", "Brier_ci_low", "Brier_ci_high",
    "O_E", "CITL", "Calibration_intercept", "Calibration_slope",
    "threshold_cal",
    "Sensitivity", "Sensitivity_ci_low", "Sensitivity_ci_high",
    "Specificity", "Specificity_ci_low", "Specificity_ci_high",
    "PPV", "PPV_ci_low", "PPV_ci_high",
    "NPV", "NPV_ci_low", "NPV_ci_high",
    "Accuracy", "TP", "FP", "TN", "FN",
]])

# Secondary one-year models.
if RUN_1Y_SECONDARY_MODELS:
    print("\n" + "=" * 100)
    print("AoU TEST PERFORMANCE — ONE-YEAR SECONDARY MODELS")
    print("=" * 100)
    display(
        performance_df.loc[
            ~performance_df["analysis_group"].eq("primary_biome_compatible")
        ][[
            "analysis_group", "model_id", "algorithm", "n_features",
            "AUROC", "AUROC_ci_low", "AUROC_ci_high",
            "AUPRC", "AUPRC_ci_low", "AUPRC_ci_high",
            "Brier", "O_E", "CITL", "Calibration_slope",
            "threshold_cal", "Sensitivity", "Specificity", "PPV", "NPV",
            "TP", "FP", "TN", "FN",
        ]]
    )

# Full calibration details with confidence intervals.
print("\n" + "=" * 100)
print("AoU TEST CALIBRATION — PRIMARY 90% POLICY MODELS")
print("=" * 100)
display(performance_df[[
    "analysis_group", "landmark", "model_id", "algorithm",
    "Brier", "Brier_ci_low", "Brier_ci_high",
    "O_E", "O_E_ci_low", "O_E_ci_high",
    "CITL", "CITL_ci_low", "CITL_ci_high",
    "Calibration_intercept", "Calibration_intercept_ci_low",
    "Calibration_intercept_ci_high",
    "Calibration_slope", "Calibration_slope_ci_low", "Calibration_slope_ci_high",
]])

# All operating policies.
print("\n" + "=" * 100)
print("AoU TEST FIXED-POLICY PERFORMANCE — 80/85/90/95% TARGET SPECIFICITY")
print("=" * 100)
display(policy_df[[
    "analysis_group", "landmark", "model_id", "algorithm",
    "target_specificity", "threshold_cal", "cal_specificity",
    "Sensitivity", "Specificity", "PPV", "NPV", "Accuracy",
    "TP", "FP", "TN", "FN",
]])

# Calibration audit.
print("\n" + "=" * 100)
print("RAW vs CALIBRATED DISCRIMINATION")
print("=" * 100)
display(raw_cal_df)

# Cost of restricting to BioMe-compatible features.
if not compare_out.empty:
    print("\n" + "=" * 100)
    print("FULL CORRECTED AoU vs BIOME-COMPATIBLE AoU MODELS")
    print("=" * 100)
    display(compare_out)

# Frozen collaborator manifest.
print("\n" + "=" * 100)
print("FROZEN MODEL MANIFEST FOR THE BIOME SCORING PACKAGE")
print("=" * 100)
display(model_manifest)

In [ ]:
import sys
import numpy as np
import pandas as pd
import scipy
import sklearn
import xgboost as xgb

print("Python:", sys.version.split()[0])
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scipy:", scipy.__version__)
print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgb.__version__)

## Interpretation and handoff rules

1. **These AoU TEST results are an internal audit, not external validation.** They show how much performance is retained when the corrected AoU models are restricted to predictors that can be reproduced in BioMe.
2. The **primary external-validation models** are the BioMe-compatible XGBoost M1–M4a/M4 models. The one-year clinical LR and M4 LR/RF models are secondary.
3. The final BioMe Python scorer should read the frozen model manifest and exact feature manifest rather than maintain hand-written model names, feature lists, or thresholds.
4. The scorer should **stop if any required feature is absent**. It should not silently fill a missing required model predictor with `NaN` or `0`.
5. BioMe may report raw and calibrated probabilities, but the primary fixed-policy metrics must use the **AoU CAL-derived thresholds unchanged**.
6. Any BioMe-native threshold analysis is exploratory only and should be clearly separated from the primary external-validation analysis.
7. Before sending the package to the collaborator, the revised BioMe R extraction script must be audited against the corrected phenotype/timing rules and against this exact feature manifest.

In [ ]:
print("=" * 100)
print("NOTEBOOK 07 COMPLETE")
print("=" * 100)
print("Key outputs:")
for path in [
    FEATURE_OUT,
    COLLAB_FEATURE_OUT,
    FEATURE_AUDIT_OUT,
    CANDIDATE_OUT,
    THRESHOLD_OUT,
    PERFORMANCE_OUT,
    POLICY_OUT,
    RAW_CAL_OUT,
    MODEL_MANIFEST_OUT,
    HYPER_OUT,
    SUMMARY_OUT,
]:
    print(" ", path)

print("\nNext step: revise and audit the BioMe R extraction + Python scoring scripts")
print("against these frozen manifests before anything is sent to the collaborator.")